In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px


CLIENT: Larry Sanders - Buyer

waterfront / 
limited budget / 
nice & isolated but central neighborhood / 
without kids (just doesn't want his kids to play with other kids because of germs)


LARRY SANDERS: FILTERING CRITERIA

Waterfront: only with a water view (waterfront == 1).

Budget: below median price (df_Larry_water['price'].quantile(0.5)).

Privacy: large neighboring lots (top 25%).

Fewer children: 2-3 bedrooms (less likely to be family-oriented).

Central: distance to the city center.

Defined city center using the coordinates (47.606° N, 122.33° W).

Selected houses with the shortest distance to center.


In [2]:
df = pd.read_csv('data/King_country_house_clean_all_ohheNaN.csv')

In [3]:
# City center coordinates (baseline for King County/Seattle area analysis)
lat_center = 47.36
long_center = -122.19

# 1. Segmenting the data: Filtering specifically for waterfront properties

df_Larry_water = df[df['waterfront'] == 1]

In [4]:
# Filter by price (exclude items priced below the median)
df_Larry_water_lim = df_Larry_water[df_Larry_water['price'] < df_Larry_water['price'].median()]

In [5]:
# Filter by lot size (keep homes in the top quartile)
df_Larry_water_lim_sq = df_Larry_water_lim[df_Larry_water_lim['sqft_lot15'] >= df_Larry_water_lim['sqft_lot15'].quantile(0.75)]

# Add a column showing the distance to the center
df_Larry_water_lim_sq['distance_to_center'] = np.sqrt(
    ((df_Larry_water_lim_sq['lat'] - lat_center) * 111) ** 2 + 
    ((df_Larry_water_lim_sq['long'] - long_center) * 85) ** 2
)

/var/folders/6g/9nftk1_n2h77z8xpm0tdmdsw0000gn/T/ipykernel_88558/1538836273.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_Larry_water_lim_sq['distance_to_center'] = np.sqrt(


In [6]:
# Check if df_central exists and recreate it if necessary
# Determine the coordinates of the city center (Seattle)
lat_center = 47.606
long_center = -122.33

# 1 Filtering specifically for waterfront properties
df_waterfront = df[df['waterfront'] == 1]

# 2. Budget constraint (we only include homes priced below the median)
df_budget_friendly = df_waterfront[df_waterfront['price'] <= df_waterfront['price'].median()]

# 3. Select houses with large adjacent lots (top 25% by sqft_lot15)
df_large_lots = df_budget_friendly[df_budget_friendly['sqft_lot15'] >= df_budget_friendly['sqft_lot15'].quantile(0.75)]

# 4. Filter by number of bedrooms (2–3 bedrooms, which reduces the likelihood of families staying there)
df_fewer_children = df_large_lots[(df_large_lots['bedrooms_num'] >= 2) & (df_large_lots['bedrooms_num'] <= 3)]

# 5. Calculating the distance to the city center
df_fewer_children['distance_to_center'] = np.sqrt(
    ((df_fewer_children['lat'] - lat_center) * 111) ** 2 + 
    ((df_fewer_children['long'] - long_center) * 85) ** 2
)

# 6.  Select houses located closest to the city center (bottom 50%)
median_distance = df_fewer_children['distance_to_center'].median()
df_central = df_fewer_children[df_fewer_children['distance_to_center'] <= median_distance]



/var/folders/6g/9nftk1_n2h77z8xpm0tdmdsw0000gn/T/ipykernel_88558/1586012063.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fewer_children['distance_to_center'] = np.sqrt(


In [13]:
df_waterfront['price'].median()

1510000.0

In [7]:
df_fewer_children.shape

(16, 25)

In [8]:
df_central.shape

(8, 25)

In [9]:
df_central

,sales_id,house_id,date,price,bedrooms_num,bathrooms_num,sqft_living,sqft_lot,floors_total,waterfront,...,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,year,month,distance_to_center
49,50,822039084,2015-03-11,1350000.0,3.0,2.50,2753.0,65005.0,1.0,1,...,1953,0,98070,47.4041,-122.451,2680.0,72513.0,2015,3,24.658258
11103,11104,7936500172,2014-05-28,1180000.0,3.0,2.50,1970.0,23180.0,1.0,1,...,1937,1998,98136,47.5495,-122.398,3030.0,34689.0,2014,5,8.528781
11412,11413,5119400075,2014-06-20,950000.0,3.0,3.25,3050.0,18892.0,1.0,1,...,1962,0,98198,47.3881,-122.326,1170.0,70973.0,2014,6,24.189290
11544,11545,2013802030,2014-09-11,357000.0,3.0,2.00,2460.0,53882.0,1.0,1,...,1955,0,98198,47.3811,-122.325,2660.0,32625.0,2014,9,24.967517
13564,13565,2623029003,2014-12-16,635000.0,3.0,1.75,1940.0,167125.0,1.0,1,...,1955,0,98070,47.4590,-122.504,1910.0,127195.0,2014,12,22.022456
14813,14814,2423029009,2014-06-17,465000.0,2.0,2.00,1494.0,19271.0,2.0,1,...,1943,1997,98070,47.4728,-122.497,1494.0,43583.0,2014,6,20.496345
17138,17139,5215200050,2014-07-29,750000.0,3.0,2.50,2960.0,69351.0,2.0,1,...,1990,0,98070,47.4000,-122.420,2350.0,41433.0,2014,7,24.111749
18023,18024,7003200120,2014-06-27,528000.0,2.0,0.75,840.0,40642.0,1.0,1,...,1937,0,98070,47.4040,-122.447,1850.0,64069.0,2014,6,24.528537


In [10]:
# Scoring: ranking the 8 filtered properties by key criteria

df_scored = df_central.copy()

# Normalize each criterion to a 0–1 scale (1 = best)
# For price and distance: lower is better → invert the scale
df_scored['score_price'] = 1 - (df_scored['price'] - df_scored['price'].min()) / (df_scored['price'].max() - df_scored['price'].min())
df_scored['score_area'] = (df_scored['sqft_living'] - df_scored['sqft_living'].min()) / (df_scored['sqft_living'].max() - df_scored['sqft_living'].min())
df_scored['score_distance'] = 1 - (df_scored['distance_to_center'] - df_scored['distance_to_center'].min()) / (df_scored['distance_to_center'].max() - df_scored['distance_to_center'].min())
df_scored['score_privacy'] = (df_scored['sqft_lot15'] - df_scored['sqft_lot15'].min()) / (df_scored['sqft_lot15'].max() - df_scored['sqft_lot15'].min())

# Weighted total score — price and distance are prioritized (35% each)
# Living area and privacy are secondary criteria (15% each)
df_scored['total_score'] = (
    df_scored['score_price'] * 0.35 +
    df_scored['score_distance'] * 0.35 +
    df_scored['score_privacy'] * 0.15 +
    df_scored['score_area'] * 0.15
)
# Display results sorted by score — highest score = best match for Larry
df_scored[['house_id', 'price', 'sqft_living', 'distance_to_center', 'sqft_lot15', 'total_score']]\
    .sort_values('total_score', ascending=False)

,house_id,price,sqft_living,distance_to_center,sqft_lot15,total_score
13564,2623029003,635000.0,1940.0,22.022456,127195.0,0.539379
11103,7936500172,1180000.0,1970.0,8.528781,34689.0,0.489890
14813,2423029009,465000.0,1494.0,20.496345,43583.0,0.468900
11544,2013802030,357000.0,2460.0,24.967517,32625.0,0.459955
17138,5215200050,750000.0,2960.0,24.111749,41433.0,0.387563
11412,5119400075,950000.0,3050.0,24.189290,70973.0,0.368381
18023,7003200120,528000.0,840.0,24.528537,64069.0,0.348949
49,822039084,1350000.0,2753.0,24.658258,72513.0,0.199694


In [71]:
import plotly.express as px

# Creating an interactive map using scatter_mapbox
fig = px.scatter_mapbox(
    df_central,
    lat="lat",
    lon="long",
    color="price",
    size="sqft_living",
    hover_name="house_id",
    hover_data=["price", "bedrooms_num", "bathrooms_num", "sqft_living", "zipcode"],
    title="Filtered Waterfront Properties for Larry",
    mapbox_style="open-street-map",
    zoom=12
)
# Increasing the size of the map window
fig.update_layout(
    width=900,   
    height=900    
)

fig.show()


Points size – living area


In [17]:
df_central['total_score'] = df_scored['total_score']

/var/folders/6g/9nftk1_n2h77z8xpm0tdmdsw0000gn/T/ipykernel_88558/836266377.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_central['total_score'] = df_scored['total_score']


In [70]:
import plotly.express as px
import plotly.graph_objects as go

fig = px.scatter_mapbox(
    df_central,
    lat="lat", lon="long",
    color="price",
    size="sqft_living",
    hover_name="house_id",
    hover_data=["price", "sqft_living", "distance_to_center", "total_score"],
    color_continuous_scale="Viridis",
    size_max=25,
    zoom=11,
    mapbox_style="carto-positron"
)

winner = df_central[df_central['house_id'] == 1110379365001722]

fig.add_trace(go.Scattermapbox(
    lat=winner['lat'],
    lon=winner['long'],
    mode='markers+text',
    marker=dict(size=25, color='red'),
    text=["★ Best match"],
    textposition="top right",
    textfont=dict(size=13, color="white"),
    name='Best match'
))

fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()

In [32]:
df_central_sorted = df_central.copy()
df_central_sorted['total_score'] = df_scored.loc[df_central_sorted.index, 'total_score']
df_central_sorted['rank'] = df_central_sorted['total_score'].rank(ascending=False).astype(int)
df_central_sorted['label'] = df_central_sorted['rank'].astype(str)

In [35]:
df_top3 = df_central_sorted[df_central_sorted['rank'] <= 3]


In [65]:


fig = go.Figure()

colors = {1: '#e63946', 2: '#f4a261', 3: '#2a9d8f'}
df_top3['color'] = df_top3['rank'].map(colors)

fig.add_trace(go.Scattermapbox(
    lat=df_top3['lat'],
    lon=df_top3['long'],
    mode='markers+text',
    marker=dict(
        size=df_top3['sqft_living'] / 80,
        color=df_top3['color'],
        opacity=0.95
    ),
    text=df_top3['rank'].astype(str),
    textfont=dict(size=13, color="black", weight="bold"),
    textposition="top center",
    hovertemplate=(
        "<b>Rank #%{customdata[0]}</b><br>"
        "Price: %{customdata[1]}<br>"
        "Living area: %{customdata[2]} sqft<br>"
        "Price/sqft: $%{customdata[3]}<br>"
        "Distance to center: %{customdata[4]:.1f} km"
        "<extra></extra>"
    ),
    customdata=df_top3[['rank', 'price_label', 'sqft_living', 'price_per_sqft', 'distance_to_center']].values,
    showlegend=False
))

fig.add_trace(go.Scattermapbox(
    lat=df_top3['lat'],
    lon=df_top3['long'],
    mode='text',
    text=df_top3['price_label'],
    textfont=dict(size=11, color="#333333"),
    textposition="bottom center",
    showlegend=False,
    hoverinfo='skip'
))

fig.add_trace(go.Scattermapbox(
    lat=[downtown['lat']],
    lon=[downtown['lon']],
    mode='markers+text',
    marker=dict(size=8, color='black'),
    text=["Downtown Seattle"],
    textposition="top right",
    textfont=dict(size=13, color="black", weight="bold"),
    showlegend=False,
    hoverinfo='skip'
))

fig.update_layout(
    mapbox_style="carto-positron",
    mapbox_zoom=10,
    mapbox_center=dict(lat=47.48, lon=-122.4),
    width=900,
    height=900,
    margin={"r":0,"t":0,"l":0,"b":0}
)

fig.show()

In [66]:
df_top3 = df_top3.sort_values('rank')

fig_card = go.Figure()

fig_card.add_trace(go.Table(
    header=dict(
        values=["", "Rank 1", "Rank 2", "Rank 3"],
        fill_color=['#f0f0f0', '#e63946', '#f4a261', '#2a9d8f'],
        font=dict(color=['black', 'white', 'white', 'white'], size=13, family="Arial"),
        align='center',
        height=40
    ),
    cells=dict(
        values=[
            ["Price", "Living area", "Price/sqft", "Distance to center", "Bedrooms", "Bathrooms"],
            [
                df_top3[df_top3['rank']==1].iloc[0]['price_label'],
                f"{int(df_top3[df_top3['rank']==1].iloc[0]['sqft_living'])} sqft",
                f"${int(df_top3[df_top3['rank']==1].iloc[0]['price_per_sqft'])}/sqft",
                f"{df_top3[df_top3['rank']==1].iloc[0]['distance_to_center']:.1f} km",
                int(df_top3[df_top3['rank']==1].iloc[0]['bedrooms_num']),
                df_top3[df_top3['rank']==1].iloc[0]['bathrooms_num']
            ],
            [
                df_top3[df_top3['rank']==2].iloc[0]['price_label'],
                f"{int(df_top3[df_top3['rank']==2].iloc[0]['sqft_living'])} sqft",
                f"${int(df_top3[df_top3['rank']==2].iloc[0]['price_per_sqft'])}/sqft",
                f"{df_top3[df_top3['rank']==2].iloc[0]['distance_to_center']:.1f} km",
                int(df_top3[df_top3['rank']==2].iloc[0]['bedrooms_num']),
                df_top3[df_top3['rank']==2].iloc[0]['bathrooms_num']
            ],
            [
                df_top3[df_top3['rank']==3].iloc[0]['price_label'],
                f"{int(df_top3[df_top3['rank']==3].iloc[0]['sqft_living'])} sqft",
                f"${int(df_top3[df_top3['rank']==3].iloc[0]['price_per_sqft'])}/sqft",
                f"{df_top3[df_top3['rank']==3].iloc[0]['distance_to_center']:.1f} km",
                int(df_top3[df_top3['rank']==3].iloc[0]['bedrooms_num']),
                df_top3[df_top3['rank']==3].iloc[0]['bathrooms_num']
            ],
        ],
        fill_color=[
            ['#f0f0f0'] * 6,
            ['#fdecea'] * 6,
            ['#fff4ec'] * 6,
            ['#edf7f5'] * 6,
        ],
        font=dict(size=12, family="Arial"),
        align='center',
        height=32
    )
))

fig_card.update_layout(
    width=700,
    height=320,
    margin={"r":20,"t":20,"l":20,"b":20}
)

fig_card.show()